In [1]:
import numpy as np
import pandas as pd
import joblib
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

SEMENTE = 42
np.random.seed(SEMENTE)

ARQUIVO_DATASET = "../metadados/etapa 2/dataset_rotulado_periodo_b.csv"
print("Ambiente configurado para o XGBoost!")

Ambiente configurado para o XGBoost!


In [2]:
# 1. Carregar dataset
df = pd.read_csv(ARQUIVO_DATASET)

# Ordenação temporal estrita
if "timestamp" in df.columns:
    df = df.sort_values("timestamp").reset_index(drop=True)

# 2. Mapeamento de classes para inteiros (exigência do XGBoost)
MAPA_CLASSES = {"OK": 0, "RISCO": 1, "FALHA": 2}
MAPA_INVERSO = {0: "OK", 1: "RISCO", 2: "FALHA"}
CLASSES_ORDEM = ["OK", "RISCO", "FALHA"]

df["target_num"] = df["classe"].map(MAPA_CLASSES)

# 3. Divisão cronológica (70% Treino, 15% Validação, 15% Teste)
total_registos = len(df)
corte_treino = int(total_registos * 0.70)
corte_val = int(total_registos * 0.85)

df_treino = df.iloc[:corte_treino].copy()
df_val = df.iloc[corte_treino:corte_val].copy()
df_teste = df.iloc[corte_val:].copy()

FEATURES_OFICIAIS = [
    "z_robusto", "aumento_pct", "jitter_relativo", "perda_pct",
    "timeout_atual", "n5_timeout", "n5_aumento80", "n5_risco"
]

X_train, y_train = df_treino[FEATURES_OFICIAIS], df_treino["target_num"]
X_val, y_val = df_val[FEATURES_OFICIAIS], df_val["target_num"]
X_test, y_test = df_teste[FEATURES_OFICIAIS], df_teste["target_num"]

print("--- DIVISÃO TEMPORAL (XGBOOST) ---")
print(f"• Treino (70%):    {len(X_train):,} amostras")
print(f"• Validação (15%): {len(X_val):,} amostras")
print(f"• Teste (15%):     {len(X_test):,} amostras (guardado)")

--- DIVISÃO TEMPORAL (XGBOOST) ---
• Treino (70%):    71,364 amostras
• Validação (15%): 15,292 amostras
• Teste (15%):     15,293 amostras (guardado)


In [3]:
# Grade de busca otimizada para o XGBoost
parametros_xgb = {
    "n_estimators": [100, 200],
    "max_depth": [3, 6, 9],
    "learning_rate": [0.01, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

# Instância do XGBClassifier para classificação multiclasse
xgb_base = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    random_state=SEMENTE,
    eval_metric="mlogloss",
    n_jobs=1
)

grid_xgb = GridSearchCV(
    estimator=xgb_base,
    param_grid=parametros_xgb,
    scoring="f1_macro",
    cv=5,
    n_jobs=-1
)

print("GridSearchCV configurado para o XGBoost!")

GridSearchCV configurado para o XGBoost!


In [4]:
print("Iniciando a busca de hiperparâmetros no XGBoost...")
grid_xgb.fit(X_train, y_train)

print("\n--- MELHORES PARÂMETROS DO XGBOOST ---")
print(grid_xgb.best_params_)

print(f"\nMelhor F1-Macro na Validação Cruzada (Treino): {grid_xgb.best_score_:.4f}")

Iniciando a busca de hiperparâmetros no XGBoost...

--- MELHORES PARÂMETROS DO XGBOOST ---
{'colsample_bytree': 1.0, 'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 200, 'subsample': 0.8}

Melhor F1-Macro na Validação Cruzada (Treino): 0.9996


In [5]:
modelo_xgb = grid_xgb.best_estimator_

# Previsão numérica e conversão de volta para os rótulos de texto
y_val_pred_num = modelo_xgb.predict(X_val)

y_val_str = [MAPA_INVERSO[val] for val in y_val]
y_val_pred_str = [MAPA_INVERSO[val] for val in y_val_pred_num]

cm = confusion_matrix(y_val_str, y_val_pred_str, labels=CLASSES_ORDEM)
df_cm = pd.DataFrame(
    cm,
    index=[f"Real_{c}" for c in CLASSES_ORDEM],
    columns=[f"Pred_{c}" for c in CLASSES_ORDEM]
)

print("--- MATRIZ DE CONFUSÃO 3x3 — XGBOOST (VALIDAÇÃO) ---")
print(df_cm)

print("\n--- RELATÓRIO DE CLASSIFICAÇÃO NA VALIDAÇÃO ---")
print(classification_report(y_val_str, y_val_pred_str, labels=CLASSES_ORDEM, digits=4))

f1_macro_val = f1_score(y_val_str, y_val_pred_str, average="macro")
print(f"\n>>> F1-SCORE MACRO NO XGBOOST (VALIDAÇÃO): {f1_macro_val:.4f} <<<")

--- MATRIZ DE CONFUSÃO 3x3 — XGBOOST (VALIDAÇÃO) ---
            Pred_OK  Pred_RISCO  Pred_FALHA
Real_OK       11926           1           0
Real_RISCO       16        3292           0
Real_FALHA        0           0          57

--- RELATÓRIO DE CLASSIFICAÇÃO NA VALIDAÇÃO ---
              precision    recall  f1-score   support

          OK     0.9987    0.9999    0.9993     11927
       RISCO     0.9997    0.9952    0.9974      3308
       FALHA     1.0000    1.0000    1.0000        57

    accuracy                         0.9989     15292
   macro avg     0.9995    0.9984    0.9989     15292
weighted avg     0.9989    0.9989    0.9989     15292


>>> F1-SCORE MACRO NO XGBOOST (VALIDAÇÃO): 0.9989 <<<


In [ ]:
# Salvar o modelo XGBoost na mesma pasta
joblib.dump(modelo_xgb, "modelos/xgboost_oficial.joblib")
print("Modelo XGBoost salvo com sucesso como 'xgboost_oficial.joblib'!")

Modelo XGBoost salvo com sucesso como 'xgboost_oficial.joblib'!
